In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# MySQL Data Factory - 批量插入工作流\n",
    "\n",
    "## 工作流程\n",
    "1. 连接数据库\n",
    "2. 查询样本记录\n",
    "3. 保存样本文件\n",
    "4. 生成待插入数据\n",
    "5. 执行批量插入"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 1️⃣ 导入模块"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "\n",
    "from src.database import DatabaseManager\n",
    "from src.data_generator import DataGenerator\n",
    "from src.bulk_inserter import BulkInserter\n",
    "from src.data_loader import DataLoader\n",
    "import pandas as pd\n",
    "from dotenv import load_dotenv\n",
    "\n",
    "print(\"✓ 模块导入成功\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 2️⃣ 连接数据库"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 加载环境变量\n",
    "load_dotenv()\n",
    "\n",
    "# 连接数据库\n",
    "db = DatabaseManager()\n",
    "db.connect()\n",
    "\n",
    "print(\"✓ 数据库连接成功\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 3️⃣ 查询样本记录"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 输入表名\n",
    "table_name = \"t_legal\"  # 修改为你要操作的表\n",
    "\n",
    "# 查询1条记录作为样本\n",
    "sample_sql = f\"SELECT * FROM {table_name} LIMIT 1\"\n",
    "sample_df = db.to_dataframe(sample_sql)\n",
    "\n",
    "print(f\"✓ 查询到样本记录：{len(sample_df)} 行\")\n",
    "print(f\"\\n样本数据预览：\")\n",
    "sample_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 4️⃣ 保存样本文件"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 保存样本到文件\n",
    "loader = DataLoader()\n",
    "sample_file = loader.save_sample(sample_df, table_name, file_type='csv')\n",
    "\n",
    "print(f\"✓ 样本已保存到：{sample_file}\")\n",
    "print(f\"\\n提示：你可以编辑 data/samples/ 目录下的文件，或参考样本生成新数据\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 5️⃣ 生成待插入数据"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 方式1：从样本生成（推荐）\n",
    "generator = DataGenerator(locale='ja_JP')\n",
    "\n",
    "# 生成数量\n",
    "count = 100  # 修改为你需要的数量\n",
    "\n",
    "# 从样本生成数据（主键会自动递增）\n",
    "primary_key = 'LEGAL_ID'  # 修改为你的主键列名\n",
    "generated_df = generator.generate_from_sample(sample_df, count=count, primary_key_col=primary_key)\n",
    "\n",
    "print(f\"✓ 生成了 {len(generated_df)} 条数据\")\n",
    "print(f\"\\n生成的数据预览：\")\n",
    "generated_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 6️⃣ 保存待插入文件"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 保存待插入数据\n",
    "pending_file = loader.save_pending(generated_df, f\"{table_name}_pending.csv\")\n",
    "\n",
    "print(f\"✓ 待插入文件已保存：{pending_file}\")\n",
    "print(f\"\\n提示：你可以编辑 data/pending/ 目录下的文件，然后执行插入\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 7️⃣ 执行批量插入"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 创建批量插入器\n",
    "inserter = BulkInserter(db)\n",
    "\n",
    "# 执行批量插入\n",
    "batch_size = 100  # 每批插入数量\n",
    "inserted_count = inserter.insert_from_dataframe(\n",
    "    generated_df, \n",
    "    table_name, \n",
    "    batch_size=batch_size\n",
    ")\n",
    "\n",
    "print(f\"\\n✓ 成功插入 {inserted_count} 条记录到 {table_name}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 8️⃣ 验证插入结果"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 验证插入结果\n",
    "count_sql = f\"SELECT COUNT(*) FROM {table_name}\"\n",
    "total_count = db.query(count_sql)[0][0]\n",
    "\n",
    "print(f\"✓ 表 {table_name} 当前总记录数：{total_count}\")\n",
    "\n",
    "# 查看最新插入的5条记录\n",
    "latest_sql = f\"SELECT * FROM {table_name} ORDER BY UPDATED_AT DESC LIMIT 5\"\n",
    "latest_df = db.to_dataframe(latest_sql)\n",
    "\n",
    "print(f\"\\n最新插入的记录：\")\n",
    "latest_df"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "### 9️⃣ 关闭连接"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# 关闭数据库连接\n",
    "db.disconnect()\n",
    "\n",
    "print(\"✓ 数据库连接已关闭\")\n",
    "print(\"\\n🎉 工作流执行完成！\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}